<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/ReidParlorCalving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cow re-ID with real farm IDs: parlor pool → calving pens

**Goal:** name the cows seen in the calving pens with their farm ID, from what they look like.

**Where the IDs come from** (no labelling by hand):
- The BoviSync parlor report for one day (`BoviSyncRecords/Nov17Parlor.xlsx`) gives every milking: cow, attach time and stall.
- A *turn* is one side of the parlor filling up. Its cows share one attach minute.
- Every cow is milked 3 times a day, with different neighbours each time.
- For each cow, the code picks the one track per milking that holds her (by turn) and that look most alike. A cow is named only when that choice is clear.
- This gives the **parlor pool**: crops with real farm IDs.

**The test:** the calving-pen cows of the days before (Nov 11–12) are named against that pool. The cows that calved those days (`Nov11To15Calvings.xlsx`) should come out. Chance level is printed next to the score.

| Cell | Step | Model | Time (T4, about) |
|---|---|---|---|
| 3 | Parlor clock: each 8-h part's start, the milkings in it, one check picture per milking | AVI chunk walk | a few min per video |
| 4 | Track + crops, parlor, only during the milkings | YOLO26m + ByteTrack, SAM 2.1 masks | ~1 h per milking |
| 5 | Track + crops, calving pens, the 2 busiest 15-min windows per video | YOLO26n + ByteTrack, SAM 2.1 | ~20 min per window |
| 6 | 8-band fingerprints of every crop | DINOv3 ViT-S/16 | 5–10 min per folder |
| 7 | Name the parlor cows → `Names/ParlorIds.csv`, `Names/Cows/<CowId>.jpg` | none (report + fingerprints) | minutes |
| 8 | Name the calving-pen tracks against the parlor pool → `Names/CalvingMatches.csv` | none | minutes |

Every cell continues where it stopped when rerun, and skips what is done. Outputs: `ReidPipelineOutputs/<Run>/`. Needs a T4 GPU for cells 4–6 and `HF_TOKEN` in Colab Secrets. Cells 3, 7 and 8 also run on a CPU runtime.

In [ ]:
# 1. Setup: mount Drive, get the code from GitHub, install the libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"
if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
import json, re
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from CalvingPipeline.Paths import BoviSyncDir, FindVideo, ReidOutputRoot
from CalvingPipeline.AviFrames import AviActivity
from CalvingPipeline.Reid.Frames import HeaderFps, LocalCopy
from CalvingPipeline.Reid.Segments import TrackAndCrop
from CalvingPipeline.Reid.Bands import EmbedBands
from CalvingPipeline.Reid.PenArea import PenAreaFor
from CalvingPipeline.Reid.Validate import LoadVideo
from CalvingPipeline.Reid.ParlorIds import (ActiveWindows, CowSheets, LoadMany, MatchCalving, MilkingFrames, NameParlor,
                                            PartOffsets, ReadCalvings, RecordedMinutes, ReadParlorReport, SessionWindows)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (fine for cells 3, 7, 8; cells 4-6 need a T4)")
try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "NOT SET - cell 6 (DINOv3) will stop; add it in Colab Secrets")

In [ ]:
# 2. Settings: the only cell to edit.
Run = "Parlor17NovCalving11Nov"          # output folder: ReidPipelineOutputs/<Run>
Year = 2026

# --- Parlor: one day of entry-parlor video + that day's BoviSync parlor report ---
ParlorVideos = ["EntryParlor17Nov.avi", "EntryParlor0217Nov.avi", "EntryParlor0317Nov.avi"]   # parts 1-3 of 17 Nov
ParlorReport = BoviSyncDir / "Nov17Parlor.xlsx"
ParlorStarts = None                      # None = the parts follow each other from 00:00 (the camera cuts a file every 8 h);
                                         # or one "HH:MM" per part, e.g. ["00:00", "08:00", "16:00"]
ParlorDetModel = "yolo26m.pt"            # medium YOLO: cows packed shoulder to shoulder

# --- Calving pens: the days to name, and the calving list ---
CalvingVideos = ["CalvingPen1Nov11Cow1.avi", "CalvingPen2Nov11Cow1.avi", "CalvingPen3Nov11Cow1.avi",
                 "CalvingPen2Nov12Cow2.avi", "CalvingPen3Nov12Cow2.avi", "CalvingPen3Nov12Cow4.avi"]
CalvingReport = BoviSyncDir / "Nov11To15Calvings.xlsx"
WindowsPerVideo, WindowMinutes = 2, 15   # the busiest stretches of each recording (a cow moving)
ExpectedDays = 1                         # a cow filmed on day D should be one that calved D-1 .. D+1

# --- Tracking and crops (cells 4-5) ---
TargetFps, EverySeconds = 5, 15          # frames looked at per second; a crop of every cow every 15 s
ImageSize, Confidence = 1024, 0.25
GateArea = 10000                         # smallest usable mask, for 1920x1080 (scaled to each video's size)

# --- Naming parlor cows (cell 7) ---
MinScore = 0.75                          # her tracks from different milkings must look at least this alike
MinMargin = 0.03                         # ... and clearly more alike than any other choice
DeleteLocalCopies = True                 # free the Colab disk after each video

RunDir = ReidOutputRoot / Run
ParlorDir, CalvingDir, NamesDir = RunDir / "Parlor", RunDir / "Calving", RunDir / "Names"
for D in (ParlorDir, CalvingDir, NamesDir):
    D.mkdir(parents=True, exist_ok=True)
Track = dict(TargetFps=TargetFps, EverySeconds=EverySeconds, ImageSize=ImageSize, Confidence=Confidence)
Report = ReadParlorReport(ParlorReport)
Calvings = ReadCalvings(CalvingReport)
def VideoDate(Name):
    M = re.search(r"CalvingPen\d+([A-Z][a-z]{2})(\d{1,2})", Name)
    return pd.Timestamp(f"{M.group(2)} {M.group(1)} {Year}")
def Drop(Video):
    if DeleteLocalCopies:
        (Path("/content/ReidVideos") / Path(Video).name).unlink(missing_ok=True)
print(f"Parlor report: {len(Report)} milkings of {Report.CowId.nunique()} cows in {Report.Session.nunique()} sessions; "
      f"calving list: {len(Calvings)} calvings. Outputs: {RunDir}")

In [ ]:
# 3. Parlor clock: when each part starts and where the milkings of the report fall in it.
#    The entry-parlor camera records all the time and cuts a file every 8 h, so part 1 = 00:00-08:00, part 2 =
#    08:00-16:00, part 3 = 16:00-24:00. Check Parlor/<video>Milkings.jpg: cows in the stalls in every picture.
Clock, Minutes = {}, []
for V in ParlorVideos:
    ActPath = ParlorDir / f"{Path(V).stem}Activity.csv"
    if ActPath.exists():
        Activity = pd.read_csv(ActPath)
    else:
        Local = LocalCopy(FindVideo(V))
        Activity = AviActivity(Local, TicksPerSecond=HeaderFps(Local))
        Activity.to_csv(ActPath, index=False)
    Minutes.append(round(RecordedMinutes(Activity)))
for V, Offset, M in zip(ParlorVideos, PartOffsets(Minutes, ParlorStarts), Minutes):
    Windows = SessionWindows(Report, Offset, M)
    Clock[V] = dict(OffsetMinutes=Offset, VideoMinutes=M,
                    Windows=Windows[["Session", "StartSeconds", "EndSeconds"]].to_dict("records"))
    Check = ParlorDir / f"{Path(V).stem}Milkings.jpg"
    if Clock[V]["Windows"] and not Check.exists():
        MilkingFrames(LocalCopy(FindVideo(V)), Clock[V]["Windows"], Check)
(ParlorDir / "Clock.json").write_text(json.dumps(Clock, indent=1))
display(pd.DataFrame([{"Video": V, "StartsAt": f"{C['OffsetMinutes'] // 60:02d}:{C['OffsetMinutes'] % 60:02d}",
                       "Hours": round(C["VideoMinutes"] / 60, 1), "Milkings": [W["Session"] for W in C["Windows"]],
                       "Windows": [f"{W['StartSeconds'] / 3600:.2f}-{W['EndSeconds'] / 3600:.2f} h" for W in C["Windows"]]}
                      for V, C in Clock.items()]))
for V in ParlorVideos:
    if (ParlorDir / f"{Path(V).stem}Milkings.jpg").exists():
        display(Image(str(ParlorDir / f"{Path(V).stem}Milkings.jpg"), width=1200))

In [ ]:
# 4. Parlor: track + crops during the milkings only (one folder per video and milking). Rerun to continue.
for V in ParlorVideos:
    for W in Clock[V]["Windows"]:
        print(f"\n=== {V}, milking {W['Session']} ===")
        TrackAndCrop(FindVideo(V), ParlorDir / f"{Path(V).stem}S{W['Session']}", StartSeconds=float(W["StartSeconds"]),
                     EndSeconds=float(W["EndSeconds"]), DetModel=ParlorDetModel, **Track)
    Drop(V)

In [ ]:
# 5. Calving pens: track + crops in the busiest windows of each recording (folders <video>Win1, Win2). Rerun to continue.
for V in CalvingVideos:
    Stem, ActPath = Path(V).stem, CalvingDir / f"{Path(V).stem}Activity.csv"
    if ActPath.exists():
        Activity = pd.read_csv(ActPath)
    else:
        Local = LocalCopy(FindVideo(V))
        Activity = AviActivity(Local, TicksPerSecond=HeaderFps(Local))
        Activity.to_csv(ActPath, index=False)
    for K, (Start, End) in enumerate(ActiveWindows(Activity, WindowsPerVideo, WindowMinutes), 1):
        print(f"\n=== {V}, window {K}: {Start / 3600:.2f}-{End / 3600:.2f} h ===")
        TrackAndCrop(FindVideo(V), CalvingDir / f"{Stem}Win{K}", StartSeconds=float(Start), EndSeconds=float(End),
                     DetModel="yolo26n.pt", **Track)
    Drop(V)

In [ ]:
# 6. 8-band fingerprints (DINOv3 ViT-S/16) of every crop in every folder. Skipped when up to date.
ParlorFolders = [ParlorDir / f"{Path(V).stem}S{W['Session']}" for V in ParlorVideos for W in Clock[V]["Windows"]]
Folders = [D for D in ParlorFolders + sorted(CalvingDir.iterdir()) if (D / "Crops.csv").exists()]
for D in Folders:
    EmbedBands(D, Token=HfToken)

In [ ]:
# 7. Name the parlor cows from the report: per cow, one track per milking that holds her and looks most alike.
#    Check Names/Cows/<CowId>.jpg: one row per milking, all rows must show the same cow.
Use = [(ParlorDir / f"{Path(V).stem}S{W['Session']}", Clock[V]["OffsetMinutes"]) for V in ParlorVideos for W in Clock[V]["Windows"]]
Use = [(D, O) for D, O in Use if (D / "Bands.npz").exists()]
PDirs, Offsets = [D for D, _ in Use], [O for _, O in Use]
PCrops, PBands = LoadMany(PDirs, Offsets, GateArea=GateArea)
Keep = (PCrops.Gated & ~PCrops.NonCow).to_numpy()
PCrops, PBands = PCrops[Keep].reset_index(drop=True), PBands[Keep]
Ids = NameParlor(PCrops, PBands, Report, 0, Width=int(PCrops.Width.iloc[0]), OutputDir=NamesDir,
                 MinScore=MinScore, MinMargin=MinMargin)
CowSheets(PCrops, Ids, NamesDir)
Wanted = Calvings[Calvings.Date.between(min(map(VideoDate, CalvingVideos)) - pd.Timedelta(days=ExpectedDays),
                                         max(map(VideoDate, CalvingVideos)) + pd.Timedelta(days=ExpectedDays))]
print(f"{Ids.CowId.nunique()} of {Report.CowId.nunique()} parlor cows named ({len(Ids)} tracks); "
      f"of the {len(Wanted)} cows that calved around the calving videos, named: {sorted(set(Wanted.CowId) & set(Ids.CowId))}")
display(Ids.groupby("CowId").agg(Milkings=("Session", "size"), Score=("Score", "first"), Margin=("Margin", "first"))
           .sort_values("Margin", ascending=False).head(20))
for Cow in sorted(set(Wanted.CowId) & set(Ids.CowId))[:6]:
    print(Cow)
    display(Image(str(NamesDir / "Cows" / f"{Cow}.jpg"), width=720))

In [ ]:
# 8. Name every calving-pen track against the parlor pool. "Expected" = the best cow is one that calved within
#    ExpectedDays of the video (the pool also holds every other cow of the pen, so guessing gives the chance level).
Named = PCrops.TrackId.isin(Ids.TrackId).to_numpy()
PoolCrops = PCrops[Named].assign(CowId=PCrops.TrackId[Named].map(Ids.set_index("TrackId").CowId)).reset_index(drop=True)
PoolBands = PBands[Named]
Rows = []
for D in sorted(D for D in CalvingDir.iterdir() if (D / "Bands.npz").exists()):
    Date = VideoDate(D.name)
    Expected = set(Calvings.CowId[(Calvings.Date - Date).abs() <= pd.Timedelta(days=ExpectedDays)])
    C, B, _ = LoadVideo(D, PenArea=PenAreaFor(D.name), GateArea=GateArea)
    Use = (C.Gated & ~C.NonCow).to_numpy()
    if not Use.any():
        continue
    M = MatchCalving(C[Use].reset_index(drop=True), B[Use], PoolCrops, PoolBands, Expected=Expected)
    M.insert(0, "Folder", D.name)
    M["Chance"] = round(len(Expected & set(PoolCrops.CowId)) / PoolCrops.CowId.nunique(), 3)
    Rows.append(M)
Matches = pd.concat(Rows, ignore_index=True)
Matches.to_csv(NamesDir / "CalvingMatches.csv", index=False)
Summary = Matches.groupby("Folder").apply(lambda G: pd.Series({
    "Tracks": len(G), "Crops": G.Crops.sum(), "TopCow": G.groupby("BestCow").Crops.sum().idxmax(),
    "TopCowShare": round(G.groupby("BestCow").Crops.sum().max() / G.Crops.sum(), 2),
    "ExpectedShare": round((G.Crops * G.Expected).sum() / G.Crops.sum(), 2), "Chance": G.Chance.iloc[0]}))
display(Summary)
print("ExpectedShare well above Chance = the calving cows are recognised in the parlor. "
      "TopCow the same in the windows of one pen and day = the same cow named the same.")

## About this notebook

Code: `CalvingPen/CalvingPipeline/Reid/ParlorIds.py` (report, clock, naming, matching), plus `Segments.py` (tracking and crops), `Bands.py` (fingerprints) and `Identity.py` (`CropScores`).

**Outputs** in `ReidPipelineOutputs/<Run>/`:
- `Parlor/Clock.json`: when each parlor video starts and its milkings; `Parlor/<video>Milkings.jpg`: a picture from the middle of each milking.
- `Parlor/<video>S<n>/`: tracks, crops and fingerprints of milking *n*.
- `Calving/<video>Win<k>/`: the same for window *k* of a calving video. *Win* = a time window of the recording.
- `Names/ParlorIds.csv`: the parlor tracks named with a farm ID (Score, Margin). `Names/ParlorCandidates.csv` has every track's possible cows.
- `Names/Cows/<CowId>.jpg`: one row per milking, to check by eye.
- `Names/CalvingMatches.csv`: each calving-pen track's best parlor cow, the runner-up, the margin, and whether the best is one that calved then.

**Limits:**
- The parlor side of the picture that holds stalls 1–28 is found by trying both.
- Tracks are split at the middle of the picture.
- The calving pens of November have no pen outline yet, so cows in neighbouring pens can appear. Those should come out as cows that did not calve.